# 第四讲 函数与可调试程序

先亲手定义函数，再把它们组合成可重跑的实验。前 7 个代码格直接编写学习版实现；后半部分导入严格模块对照。

先修：第 3 讲的变量、列表、条件与循环。环境：Python 3.10+ 与 Jupyter，普通 CPU、离线、无 API。请从本讲目录打开，保留 experiment.py 和 data/。

运行方法：逐格先预测输出，再执行。最终重启内核并全部运行。所有故意失败案例只捕获预期的异常，所以整份参考 Notebook 应可顺序成功执行。主实验会更新 outputs/result.json；故障文件与测试报告只在临时目录保存。

配套 lab.md 给出未完成骨架、修改任务与验收要求。这份 Notebook 是可运行参考答案，不应代替自己编写。

## 1 亲手定义一个预测函数

def 只建立定义，调用才计算；return 把结果交回调用处。参数顺序是面积、斜率、截距。关键字调用明确各个名字。

练习：先自己写同样功能，再对照。此格尚未验证类型，严格契约稍后添加。

In [1]:
def my_predict(area, weight, bias):
    prediction = weight * area + bias
    return prediction

estimate = my_predict(70, 2, 10)
assert estimate == 150
assert my_predict(bias=10, area=70, weight=2) == 150
print("预测万元：", estimate)

预测万元： 150


## 2 显示了数值 不代表返回了数值

下面函数没有 return，默认返回 None。输出中的第一次 150 来自函数内部的 print，第二行是调用者实际接到的结果。

In [2]:
def print_only(area, weight, bias):
    print(weight * area + bias)

printed_result = print_only(70, 2, 10)
assert printed_result is None
print("调用者得到：", printed_result)

150
调用者得到： None


## 3 参数与外层名字分开

外层同名变量不替代这次调用明确传入的参数。hidden_predict 则展示隐藏依赖：它读取外层权重，外部改变后结果跟着变。正式预测必须使用 my_predict。

In [3]:
area = 999
weight = 2
bias = 10

def hidden_predict(area):
    return weight * area + bias

before = hidden_predict(70)
weight = 3
after = hidden_predict(70)
assert [before, after] == [150, 220]
assert my_predict(70, 2, 10) == 150
assert area == 999
print("隐藏依赖前后：", before, after)
print("明确传参：", my_predict(70, 2, 10), "外层面积：", area)

隐藏依赖前后： 150 220
明确传参： 150 外层面积： 999


## 4 自己把累计循环变成 MAE

先手算绝对误差 4、6，平均 5。raise 主动拒绝不符合约定的长度。这里是便于初读的骨架，尚不涵盖错误元素类型和非有限数值。

修改练习：把 total = 0 放到循环内部，预测会得到多少，再修回；最后保留正确版本。

In [4]:
def my_mae(predictions, labels):
    if len(predictions) == 0:
        raise ValueError("至少需要一个样本")
    if len(predictions) != len(labels):
        raise ValueError("预测与标签长度必须相同")
    total = 0
    for index in range(len(predictions)):
        total = total + abs(predictions[index] - labels[index])
    return total / len(predictions)

assert my_mae([140, 180], [144, 174]) == 5
assert my_mae([140], [144]) == 4
print("两样本 MAE：", my_mae([140, 180], [144, 174]))
print("单样本 MAE：", my_mae([140], [144]))
try:
    my_mae([], [])
except ValueError:
    print("正确拒绝空列表")
else:
    raise AssertionError("空列表本应失败")

两样本 MAE： 5.0
单样本 MAE： 4.0
正确拒绝空列表


## 5 给预测函数补上严格数值契约

只接受普通 int/float，排除 bool，转为有限 float，并再次检查运算结果。type 检查数据类型，math.isfinite 检查 NaN 与无穷。极大整数可能不能转成 float，明确拒绝。

as error 暂时保存异常对象；from error 保留原始原因。预测函数不替调用者猜测字符串单位或格式。

In [5]:
import math

def my_number(value, name):
    if type(value) not in (int, float):
        raise TypeError(name + " 必须为普通 int/float")
    try:
        number = float(value)
    except OverflowError as error:
        raise ValueError(name + " 超出 float 范围") from error
    if not math.isfinite(number):
        raise ValueError(name + " 必须有限")
    return number


def my_checked_predict(area, weight, bias):
    area = my_number(area, "area")
    weight = my_number(weight, "weight")
    bias = my_number(bias, "bias")
    result = weight * area + bias
    return my_number(result, "result")

assert my_checked_predict(70, 2, 10) == 150
for bad_value in ["70", True]:
    try:
        my_checked_predict(bad_value, 2, 10)
    except TypeError:
        print("正确拒绝类型：", type(bad_value).__name__)
    else:
        raise AssertionError("错误类型本应失败")
try:
    my_checked_predict(float("nan"), 2, 10)
except ValueError:
    print("正确拒绝 NaN")
else:
    raise AssertionError("NaN 本应失败")

正确拒绝类型： str
正确拒绝类型： bool
正确拒绝 NaN


## 6 给 MAE 补上严格数值契约

容器必须是列表或元组；元组写成 (140,180)，可按位置读取。使用自己刚写的 my_number 检查每项以及累计。返回值和标签单位相同。

这仍是简单逐项求和：极大误差的中间累计溢出会失败，即使数学平均有限。本讲没有承诺任意精度。

In [6]:
def my_checked_mae(predictions, labels):
    if type(predictions) not in (list, tuple) or type(labels) not in (list, tuple):
        raise TypeError("需要列表或元组")
    if len(predictions) == 0:
        raise ValueError("至少需要一个样本")
    if len(predictions) != len(labels):
        raise ValueError("预测与标签长度必须相同")
    total = 0.0
    for index in range(len(predictions)):
        prediction = my_number(predictions[index], "prediction")
        label = my_number(labels[index], "label")
        error = my_number(abs(prediction - label), "error")
        total = my_number(total + error, "total")
    return my_number(total / len(predictions), "MAE")

assert my_checked_mae([140, 180], [144, 174]) == 5
assert my_checked_mae((140,), (144,)) == 4
assert my_checked_mae([240, 280], [244, 274]) == 5
assert my_checked_mae([180, 140], [174, 144]) == 5
try:
    my_checked_mae([1], [1, 2])
except ValueError:
    print("正确拒绝不等长")
else:
    raise AssertionError("不等长本应失败")
print("严格 MAE 与两个不变性检查通过")

正确拒绝不等长
严格 MAE 与两个不变性检查通过


## 7 字典给每个数值一个字段名

record 是一条记录，records 是记录组成的列表。predict_with_ids 是你自己实现的一个小型批量函数，返回新列表，不修改原始记录。这里的原始记录由我们手工确认；完整模块还验证结构。

验收：写出返回列表中每个字段的含义，再修改一条面积预测变化。

In [7]:
record = {"id": "T1", "area_m2": 65}
records = [record, {"id": "T2", "area_m2": 85}]

def predict_with_ids(input_records, weight, bias):
    result = []
    for row in input_records:
        result.append({"id": row["id"],
                       "prediction_wan": my_checked_predict(row["area_m2"], weight, bias)})
    return result

manual_predictions = predict_with_ids(records, 2, 10)
assert manual_predictions[0]["prediction_wan"] == 140
assert record == {"id": "T1", "area_m2": 65}
print(manual_predictions)

[{'id': 'T1', 'prediction_wan': 140.0}, {'id': 'T2', 'prediction_wan': 180.0}]


## 8 导入严格模块并看清 CSV 先读出文本

import 只加载 experiment.py 的定义，主入口保护使它不会自动写结果。Path(experiment.__file__) 用模块文件位置找随课数据。

with 管理文件关闭；DictReader 按表头建立字典；list 收集各行。随后 load_table 才执行类型转换和完整验证。

In [8]:
import csv
from pathlib import Path
import experiment

unit_dir = Path(experiment.__file__).resolve().parent
with (unit_dir / "data" / "train.csv").open(encoding="utf-8", newline="") as handle:
    raw_rows = list(csv.DictReader(handle))
train = experiment.load_table(unit_dir / "data" / "train.csv", "train")
assert type(raw_rows[0]["area_m2"]) is str
assert type(train[0]["area_m2"]) is float
assert len(train) == 4
assert my_checked_predict(70, 2, 10) == experiment.predict_one(70, 2, 10)
assert my_checked_mae([140, 180], [144, 174]) == experiment.mean_absolute_error([140, 180], [144, 174])
print("转换前后类型：", type(raw_rows[0]["area_m2"]).__name__, type(train[0]["area_m2"]).__name__)
print("第一条训练记录：", train[0])

转换前后类型： str float
第一条训练记录： {'id': 'H1', 'area_m2': 50.0, 'price_wan': 110.0}


## 9 先自己选择两个候选 再核对完整选择

select_two 只接收训练记录。用循环构造预测与标签，再调用自己的 MAE，返回得分较小的名字；平分保留 A。先自行实现再对照。

完整模块把它推广为三候选列表，仍然只接收训练数据。

In [9]:
def select_two(train_records):
    a_predictions = []
    b_predictions = []
    labels = []
    for row in train_records:
        a_predictions.append(my_checked_predict(row["area_m2"], 2, 0))
        b_predictions.append(my_checked_predict(row["area_m2"], 2, 10))
        labels.append(row["price_wan"])
    a_score = my_checked_mae(a_predictions, labels)
    b_score = my_checked_mae(b_predictions, labels)
    if b_score < a_score:
        return "B"
    return "A"

assert select_two(train) == "B"
fitted = experiment.fit_candidates(train, experiment.default_candidates())
assert fitted["best_model"]["name"] == "B"
training_scores = []
for item in fitted["candidate_scores"]:
    training_scores.append(item["train_mae_wan"])
assert training_scores == [10, 0, 7.5]
print("训练分数：", fitted["candidate_scores"])
print("选中：", fitted["best_model"])

训练分数： [{'name': 'A', 'train_mae_wan': 10.0}, {'name': 'B', 'train_mae_wan': 0.0}, {'name': 'C', 'train_mae_wan': 7.5}]
选中： {'name': 'B', 'weight': 2.0, 'bias': 10.0}


## 10 先预测 再按编号评价

test_inputs 没有真实价格，predict_records 只接受编号与面积。之后才打开标签文件。即使标签行顺序相反，evaluate 按 id 对齐仍得到同一结果。

In [10]:
inputs = experiment.load_table(unit_dir / "data" / "test_inputs.csv", "test_inputs")
predictions = experiment.predict_records(fitted["best_model"], inputs)
labels = experiment.load_table(unit_dir / "data" / "test_labels.csv", "test_labels")
evaluation = experiment.evaluate(predictions, labels)
reversed_labels = [labels[1], labels[0]]
reordered_evaluation = experiment.evaluate(predictions, reversed_labels)
assert evaluation["mae_wan"] == 5
assert reordered_evaluation["mae_wan"] == 5
print("按编号评价：", evaluation)

按编号评价： {'mae_wan': 5.0, 'rows': [{'id': 'T1', 'prediction_wan': 140.0, 'label_wan': 144.0, 'absolute_error_wan': 4.0}, {'id': 'T2', 'prediction_wan': 180.0, 'label_wan': 174.0, 'absolute_error_wan': 6.0}]}


## 11 合法数值也可能配错对象

下面故意忽视编号，只按当前行位置比较反序标签，得到 35。错误发生在配对，MAE 公式照常执行。修复是恢复身份对应，不能修改 MAE 公式来凑出 5。

In [11]:
wrong_predictions = [predictions[0]["prediction_wan"], predictions[1]["prediction_wan"]]
wrong_labels = [reversed_labels[0]["price_wan"], reversed_labels[1]["price_wan"]]
wrong_mae = my_checked_mae(wrong_predictions, wrong_labels)
assert wrong_mae == 35
print("错误按位置评价：", wrong_mae, "正确按编号评价：", evaluation["mae_wan"])

错误按位置评价： 35.0 正确按编号评价： 5.0


## 12 三种故障只捕获各自预期的异常

正常执行反而进入 else 判失败，这样“本来应错却没错”不会漏检。TemporaryDirectory 是临时工作区，用完自动清理。路径故障只输出错误类型，避免将自己的完整目录保存在公开 Notebook。

In [12]:
from tempfile import TemporaryDirectory

try:
    values = [10, 20]
    values[2]
except IndexError:
    print("预期 IndexError：长度为 2 的列表没有位置 2")
else:
    raise AssertionError("应触发索引错误")

try:
    experiment.predict_one("70", 2, 10)
except TypeError:
    print("预期 TypeError：文本不是已验证的数值")
else:
    raise AssertionError("应触发类型错误")

with TemporaryDirectory() as temporary:
    try:
        experiment.load_table(Path(temporary) / "missing.csv", "train")
    except FileNotFoundError:
        print("预期 FileNotFoundError：文件不存在，不回退旧数据")
    else:
        raise AssertionError("应触发路径错误")

预期 IndexError：长度为 2 的列表没有位置 2
预期 TypeError：文本不是已验证的数值
预期 FileNotFoundError：文件不存在，不回退旧数据


## 13 故障 CSV 不污染原数据

只在临时目录写入重复编号和非有限面积。两个文件都必须被拒绝；不跳过坏行，不填零。try/except 只捕获预期 ValueError，其他意外继续暴露。

In [13]:
bad_csv_texts = [
    "id,area_m2,price_wan\nH1,50,110\nH1,60,130\n",
    "id,area_m2,price_wan\nH1,nan,110\n",
]
with TemporaryDirectory() as temporary:
    for index in range(len(bad_csv_texts)):
        path = Path(temporary) / ("bad_" + str(index) + ".csv")
        path.write_text(bad_csv_texts[index], encoding="utf-8")
        try:
            experiment.load_table(path, "train")
        except ValueError as error:
            print("正确拒绝：", error)
        else:
            raise AssertionError("坏 CSV 本应失败")

正确拒绝： bad_0.csv: train record 2 has duplicate id H1
正确拒绝： bad_1.csv line 2 area_m2 must be finite


## 14 运行整条实验并读回 JSON

result 同时是返回值和保存内容。json.loads 读回文本，按结构比较；数字不变成带单位字符串。每次会更新本讲 outputs/result.json。

基线 150 来自四条训练价格的均值，不由测试标签决定。

In [14]:
import json

result = experiment.run_experiment(unit_dir / "data", unit_dir / "outputs")
loaded = json.loads((unit_dir / "outputs" / "result.json").read_text(encoding="utf-8"))
assert result == loaded
assert result["best_model"]["name"] == "B"
assert result["test_mae_wan"] == 5
assert result["baseline_prediction_wan"] == 150
assert result["baseline_test_mae_wan"] == 15
assert str(unit_dir) not in json.dumps(result)
print("JSON 读回一致；B；测试 MAE 5；训练均值基线 MAE 15")
filenames = []
for item in result["input_files"]:
    filenames.append(item["filename"])
print("输入文件名：", filenames)

JSON 读回一致；B；测试 MAE 5；训练均值基线 MAE 15
输入文件名： ['train.csv', 'test_inputs.csv', 'test_labels.csv']


## 15 新进程与另一个工作目录

subprocess.run 启动另一个 Python 进程，sys.executable 选用当前 Python，cwd 指定新进程的工作目录。参数按列表逐项传入，不需要拼接 shell 命令。

绝对路径只在运行时计算以启动文件，结果不打印或保存这些私人位置。临时目录用完清理。

In [15]:
import subprocess
import sys

with TemporaryDirectory() as temporary:
    temporary_path = Path(temporary)
    process = subprocess.run(
        [sys.executable, str(unit_dir / "experiment.py"),
         "--output-dir", str(temporary_path / "out")],
        cwd=temporary_path, text=True, capture_output=True, check=True)
    cli_summary = json.loads(process.stdout)
    cli_result = json.loads((temporary_path / "out" / "result.json").read_text(encoding="utf-8"))
    assert cli_summary == {"best_model": "B", "test_mae_wan": 5.0, "baseline_test_mae_wan": 15.0}
    assert cli_result == result
    print("不同工作目录 CLI 与当前结果一致：", cli_summary)

不同工作目录 CLI 与当前结果一致： {'best_model': 'B', 'test_mae_wan': 5.0, 'baseline_test_mae_wan': 15.0}


## 16 运行独立检查清单

先读 lab.md 的测试器解释：check 比较已知答案，expect_error 只接住指定异常，函数及参数列表通过 function(*arguments) 调用。清单还测试 malformed CSV、非有限数、溢出、编号错配、测试标签不能改变训练选择、-O 不移除必要验证。

报告存入临时目录，避免本 Notebook 反复覆盖随课测试证据；摘要会保留在输出中。

In [16]:
import test_experiment

with TemporaryDirectory() as temporary:
    report = test_experiment.run_tests(Path(temporary) / "test-result.json")
    assert report["status"] == "passed"
    assert report["check_count"] == 122
    print("验收状态：", report["status"], "检查数量：", report["check_count"])
print("全部参考实验完成。下一步：关掉答案，独立完成 lab.md 的五项验收。")

PASS: 122 explicit checks
验收状态： passed 检查数量： 122
全部参考实验完成。下一步：关掉答案，独立完成 lab.md 的五项验收。


## 完成后的解释题

1. 为什么只打印 150 的函数不能替代返回 150 的函数？
2. 为什么两个等长数值列表仍可能代表错误配对？
3. 为什么传入测试标签的模型选择函数值得警惕？
4. 为什么真实数据清洗不能通过 except Exception: pass 实现？
5. 为什么本次 122 项程序检查不能证明房价模型具有现实泛化能力？

参照讲义与实验指南作答。所有数据为原创手工合成；官方语义来源及核验日期见 source-checks.json。